Name : Amardeep Kumar

Roll No : DA25M502

# **Question 10 :** Find the advantages of using NumPy or PyTorch over naive python iterators and operations while working with neural networks.

---

You are required to implement a fully-connected feedforward neural network with a single hidden layer (4-neurons) with ReLU activation, one output layer (one neuron) with sigmoid activation, using three different approaches:
1. Pure Python using iterators
2. NumPy arrays
3. PyTorch tensors

You will then compare their execution speed as a function of input dimension for a single data-point.

Repeat it for a batched-input and explore how broadcasting is done in NumPy or PyTorch, and explicitly mention where can it be used in the given scenario (which operation during the forward pass can utilize the automatic-broadcasting?).

Assume the dimensions wherever required.

**Bonus:** Compare the speedup in PyTorch when tensors are in the CPU and when tensors are in the GPU (use Google Colab or Kaggle notebooks for free GPU access) and find the reasons for any unexpected behavior if encountered.

In [ ]:
#imports
import time
import math
import random
import numpy as np
import torch

## Python Iterators + NumPy + PyTorch

In [ ]:
#seed setup
np.random.seed(42)
torch.manual_seed(42)
random.seed(42)

# activation function#
def relu(x):
    return max(0, x)

def sigmoid(x):
    return 1 / (1 + math.exp(-x))

def relu_np(x):
    return np.maximum(0, x)

def sigmoid_np(x):
    return 1 / (1 + np.exp(-x))


# using pure python
def forward_python(x, W1, b1, W2, b2):
    # Hidden layer (4 neurons)
    h = []
    for i in range(4):
        s = 0
        for j in range(len(x)):
            s += W1[i][j] * x[j]
        s += b1[i]
        h.append(relu(s))

    # Output layer
    o = 0
    for i in range(4):
        o += W2[i] * h[i]
    o += b2

    return sigmoid(o)


#using numpy
def forward_numpy(x, W1, b1, W2, b2):
    h = relu_np(W1 @ x + b1)
    y = sigmoid_np(W2 @ h + b2)
    return y


#using pytrorch
def forward_torch(x, W1, b1, W2, b2):
    h = torch.relu(W1 @ x + b1)
    y = torch.sigmoid(W2 @ h + b2)
    return y

## Data and Output Validation

In [ ]:
# data and weight for test
d = 100  # input dimension

x_np = np.random.randn(d)
x_list = x_np.tolist()
x_torch = torch.tensor(x_np, dtype=torch.float32)

W1_np = np.random.randn(4, d)
b1_np = np.random.randn(4)
W2_np = np.random.randn(4)
b2_np = np.random.randn()

W1_torch = torch.tensor(W1_np, dtype=torch.float32)
b1_torch = torch.tensor(b1_np, dtype=torch.float32)
W2_torch = torch.tensor(W2_np, dtype=torch.float32)
b2_torch = torch.tensor(b2_np, dtype=torch.float32)


#validation
print("\nOutput:")
print("="*50)
y_py = forward_python(x_list, W1_np.tolist(), b1_np.tolist(),
                      W2_np.tolist(), b2_np)
y_np = forward_numpy(x_np, W1_np, b1_np, W2_np, b2_np)
y_torch = forward_torch(x_torch, W1_torch, b1_torch, W2_torch, b2_torch)

print("Python Output :", y_py)
print("NumPy Output  :", y_np)
print("Torch Output  :", y_torch.item())


Output:
Python Output : 6.665285562963093e-05
NumPy Output  : 6.665285562963093e-05
Torch Output  : 6.665286491625011e-05


## Speed Test on Single Input

In [ ]:
# speed test on single input
print("\nSpeed Test: Single Input  :")
print("="*50)

N = 1000

start = time.time()
for _ in range(N):
    forward_python(x_list, W1_np.tolist(), b1_np.tolist(),
                   W2_np.tolist(), b2_np)
print("Python Time:", time.time() - start)

start = time.time()
for _ in range(N):
    forward_numpy(x_np, W1_np, b1_np, W2_np, b2_np)
print("NumPy Time :", time.time() - start)

start = time.time()
for _ in range(N):
    forward_torch(x_torch, W1_torch, b1_torch, W2_torch, b2_torch)
print("Torch CPU Time:", time.time() - start)


Speed Test: Single Input  :
Python Time: 0.03049159049987793
NumPy Time : 0.006238460540771484
Torch CPU Time: 0.015584707260131836


## Batch Input + Broadcasting

In [ ]:
#batch input + Broadcasting
print("\nBatched Input Test :")
print("="*50)

batch_size = 512
X_batch = np.random.randn(batch_size, d)

# NumPy batch forward
H = relu_np(X_batch @ W1_np.T + b1_np)   # broadcasting b1
Y = sigmoid_np(H @ W2_np + b2_np)        # broadcasting b2

print("NumPy batch output shape:", Y.shape)

# PyTorch batch forward
X_batch_torch = torch.tensor(X_batch, dtype=torch.float32)

H_t = torch.relu(X_batch_torch @ W1_torch.T + b1_torch)
Y_t = torch.sigmoid(H_t @ W2_torch + b2_torch)

print("Torch batch output shape:", Y_t.shape)


Batched Input Test :
NumPy batch output shape: (512,)
Torch batch output shape: torch.Size([512])


## GPU TEST

In [ ]:
#gpu test
if torch.cuda.is_available():
    print("\nGPU Test:")
    print("="*50)
    device = torch.device("cuda")

    x_gpu = x_torch.to(device)
    W1_gpu = W1_torch.to(device)
    b1_gpu = b1_torch.to(device)
    W2_gpu = W2_torch.to(device)
    b2_gpu = b2_torch.to(device)

    for _ in range(10):
        forward_torch(x_gpu, W1_gpu, b1_gpu, W2_gpu, b2_gpu)

    torch.cuda.synchronize()

    start = time.time()
    for _ in range(N):
        forward_torch(x_gpu, W1_gpu, b1_gpu, W2_gpu, b2_gpu)
    torch.cuda.synchronize()

    print("Torch GPU Time:", time.time() - start)
else:
    print("\nGPU not available")


GPU Test:
Torch GPU Time: 0.06840729713439941
